# Train Attribute Heads (Occasion/Fit/Material) + A/B Evaluation

This notebook trains the remaining 3 multi-head projections using the **iMaterialist Fashion
Attribute Dataset** (1M+ images, 228 expert-curated attributes across 8 groups), then
evaluates the full system against the baseline.

**Dataset:** iMaterialist Fashion (FGVC5 @ CVPR 2018)
- 1,012,947 training images + 9,897 validation images
- 34 Material labels (silk, denim, leather, ...) → `material_head`
- 21 Style labels (casual, formal, sporty, ...) → `occasion_head`
- 16 Sleeve + Neckline labels → `fit_head`

**Prerequisites:** compat_head and style_head already trained (from notebook 1).

**Sections:**
1. Setup & clone repo
2. Download iMaterialist annotations + images
3. Parse attributes & build contrastive pairs
4. Pre-compute DINOv2 embeddings
5. Train occasion/fit/material heads
6. A/B evaluation (FITB + Compatibility AUC)

## 1. Setup

In [ ]:
!git clone https://github.com/anushreeberlia/loom.git
%cd /content/loom
!pip install -q -r train/requirements.txt
!pip install -q gdown scikit-learn

import os
os.makedirs("data/imaterialist/images", exist_ok=True)

In [ ]:
# If you already have the repo cloned, just pull latest
# %cd /content/loom
# !git pull

## 2. Download iMaterialist Fashion Dataset

**iMaterialist Fashion Attribute Dataset** (FGVC5 @ CVPR 2018):
- 1M+ images with 228 fine-grained attributes across 8 groups
- Annotations are JSON files (image URLs + multi-label annotations)
- Images hosted by Wish (downloaded via URLs in the JSON)

Step 1: Download annotation JSONs (~200MB) from Google Drive
Step 2: Download a subset of images (~60K) for training

In [ ]:
# Download annotation JSONs from Google Drive (train.json ~150MB, val.json ~2MB)
!gdown "https://drive.google.com/uc?id=1oh_GDZY2IQwB_eKCV1ZbWiXkVe5WGEG-" -O data/imaterialist/train.json
!gdown "https://drive.google.com/uc?id=11FiOABXkkidTZbNse1zg6HnqLay_0XL5" -O data/imaterialist/val.json

# Verify
!ls -la data/imaterialist/*.json

In [ ]:
# Quick sanity check: peek at the JSON structure
import json

with open("data/imaterialist/train.json") as f:
    data = json.load(f)

print(f"Images: {len(data['images'])}")
print(f"Annotations: {len(data['annotations'])}")
print(f"\nSample image: {data['images'][0]}")
print(f"Sample annotation: {data['annotations'][0]}")

### Download images (subset)

We don't need all 1M images. Download ~60K images that have material/style/fit labels.
This takes ~20-30 min with 32 parallel workers. Images are ~5-50KB each from Wish CDN.

In [ ]:
# Download images (60K subset focused on material/style/fit labels)
# This downloads from Wish CDN URLs in the annotation JSON
!python train/data/prepare_imaterialist.py \
    --data-dir data/imaterialist \
    --download-images \
    --max-images 60000

# Check how many we got
!find data/imaterialist/images/ -name "*.jpg" | wc -l

## 3. Parse Attributes & Build Contrastive Pairs

The iMaterialist dataset has 228 labels across 8 groups. We map them to our heads:
- **Material** (label_ids 129-162, 34 classes): nylon, silk, denim, leather, velvet, ...
- **Style → Occasion** (label_ids 207-227, 21 classes): casual, formal, sporty, party, ...
- **Sleeve + Neckline → Fit** (label_ids 163-173 + 202-206, 16 classes): long/short sleeve, v-neck, ...

Positive pair = two images sharing at least one label in the group.
Negative pair = two images with NO shared labels in the group.

In [ ]:
# Build contrastive pairs from iMaterialist annotations
# (annotations were already loaded during image download step above;
#  this reruns just pair generation if needed)
!python train/data/prepare_imaterialist.py \
    --data-dir data/imaterialist \
    --neg-ratio 2 \
    --max-positive 300000

In [ ]:
# Verify pair files
!ls -la data/imaterialist/*.csv
!wc -l data/imaterialist/*_pairs.csv

## 4. Pre-compute DINOv2 Embeddings

Run all downloaded iMaterialist images through frozen DINOv2 backbone.
~60K images at batch_size=64 on T4 GPU takes ~15-20 minutes.

In [ ]:
!python train/precompute_backbones.py --data-dir data/imaterialist --batch-size 64 --no-segment

In [ ]:
# Verify embeddings file
import h5py
with h5py.File('data/imaterialist/backbone_embeddings.h5', 'r') as f:
    print(f'Embeddings shape: {f["embeddings"].shape}')
    print(f'Items: {len(f["item_ids"])}')

## 5. Train Attribute Heads

Each head trains with InfoNCE contrastive loss on attribute-based pairs.
~10 min per head on T4.

In [ ]:
# Train occasion head (from iMaterialist "style" group: casual, formal, sporty, party, ...)
!python train/train_heads.py --config train/config_imaterialist.yaml --head occasion

In [ ]:
# Train fit head (from sleeve + neckline groups: long/short sleeve, v-neck, turtleneck, ...)
!python train/train_heads.py --config train/config_imaterialist.yaml --head fit

In [ ]:
# Train material head (34 classes: silk, denim, leather, velvet, lace, mesh, knit, ...)
!python train/train_heads.py --config train/config_imaterialist.yaml --head material

In [ ]:
# Verify all weights saved
!ls -la models/multihead/

## 6. A/B Evaluation

Compare the trained multi-head system vs raw DINOv2 cosine baseline.

Uses Polyvore test data (need to have run notebook 1 first for Polyvore embeddings).

In [ ]:
# Download Polyvore test data if not already present
import os
if not os.path.exists('data/polyvore/backbone_embeddings.h5'):
    print("Need Polyvore embeddings for A/B eval.")
    print("Run notebook 1 first, or copy backbone_embeddings.h5 from that session.")
else:
    print(f"Polyvore embeddings found: {os.path.getsize('data/polyvore/backbone_embeddings.h5') / 1e6:.1f} MB")

In [ ]:
# Run A/B evaluation (uses Polyvore test outfits to compare systems)
!python train/eval_ab_test.py --config train/config.yaml --data-dir data/polyvore

## 7. Download Trained Weights

Download all trained head weights to use in production.

In [ ]:
from google.colab import files

# Package all weights
!tar czf multihead_weights.tar.gz models/multihead/
!ls -la multihead_weights.tar.gz

# Download to your machine
files.download('multihead_weights.tar.gz')

## Usage

After downloading `multihead_weights.tar.gz`:

```bash
cd loom/
tar xzf ~/Downloads/multihead_weights.tar.gz
# Weights are now in models/multihead/ -- production code auto-loads them
```

The inference code in `services/multihead.py` will automatically detect and load
trained weights on next server start.